# Agentic RFP Evaluation

**Project Purpose**: Evaluate Request for Proposals (RFPs) using autonomous agent workflows to produce accurate, objective rankings.

## High-Level Workflow
Setup → Input → Batch → Evaluate → Validate → Score → Benchmark → Rank → Persist → Present

## Major Components
- Orchestrator Agent
- Document Tool
- Evaluation Agent
- Validation Tool
- Ranking Tool
- SQLite database
- Streamlit UI



## Phase 1 — Foundations

SQLite schema, seeded criteria, shared Pydantic models. Run `python database.py` once to create + seed `data/rfp_eval.db`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import database as db
from models import Criterion, EXPERIENCE_RATING_MIN, EXPERIENCE_RATING_MAX

db.setup_database()  # idempotent: init_schema + seed_criteria
print('Experience rating scale:', EXPERIENCE_RATING_MIN, '..', EXPERIENCE_RATING_MAX)

In [ ]:
criteria = db.get_active_criteria()
for c in criteria:
    print(f'[{c.criterion_id}] {c.name:22s} w={c.weight:5.1f}  max={c.max_score}  | {c.description}')
print('Active weight total:', db.active_weight_total())
assert sum(c.weight for c in criteria) == 100

In [ ]:
# Criteria CRUD is add / update / set_active only — there is no hard delete.
print('has delete_criterion?', hasattr(db, 'delete_criterion'))
print('run in progress?', db.is_run_in_progress())

## Phase 2 — Document Tool & Evaluation Agent

`document_tool.extract_text` pulls raw text from a supplier PDF (PyMuPDF only, typed
`DocumentExtractionError` on empty/corrupt files). `evaluation_agent.evaluate_supplier`
runs the LangChain chain (`ChatPromptTemplate | ChatGoogleGenerativeAI | PydanticOutputParser`)
against the run's frozen criterion snapshot, returning a parsed `LLMEvaluationOutput`.

Requires `GEMINI_API_KEY` in the environment and supplier PDFs under `data/sample_pdfs/`.

In [ ]:
import glob
from dotenv import load_dotenv

import document_tool
import evaluation_agent

load_dotenv(os.path.abspath('../.env'))

snapshot = db.get_active_criteria()  # in the real run the orchestrator freezes this once at Batch time
pdfs = sorted(glob.glob(os.path.abspath('../data/sample_pdfs/*.pdf')))
print('sample PDFs found:', [os.path.basename(p) for p in pdfs])

if pdfs:
    path = pdfs[0]
    name = os.path.splitext(os.path.basename(path))[0]
    text = document_tool.extract_text(path, name)
    print(f'{name}: extracted {len(text)} chars')

    result = evaluation_agent.evaluate_supplier(name, text, snapshot)
    print('supplier:', result.supplier_name)
    print('criteria returned:', len(result.criteria), 'of', len(snapshot), 'active')
    for cr in result.criteria:
        print(f'  id={cr.criterion_id} score={cr.score}/{cr.max_score} :: {cr.justification[:80]}')
    print('risks:', result.risks)
    print('summary:', result.overall_summary[:200])
    assert len(result.criteria) == len(snapshot)
else:
    print('Add supplier PDFs to data/sample_pdfs/ to smoke-test the evaluation chain.')

## Phase 3 — Validation & Scoring

`validation_tool.normalize_llm_output` turns a raw (possibly malformed) `LLMEvaluationOutput`
into exactly one trustworthy `CriterionResult` per active criterion plus a list of warnings.
`scoring.py` then computes the deterministic arithmetic — absolute weighted score, per-run
benchmarks, gap, relative performance %, and PPI — all in plain Python on the frozen snapshot.

The cell below exercises Edge Case Checklist items **2–7** with a hand-computed fixture.

In [ ]:
from models import CriterionResult, LLMEvaluationOutput
from validation_tool import normalize_llm_output, MALFORMED_POLICY_WARNING
from scoring import compute_benchmarks, score_supplier

snap = db.get_active_criteria()  # 5 seeded criteria: weights 30/20/20/20/10, max 10 each

# Supplier A — edge cases 3 (clip 12->10), 2 (crit 4 missing -> 0), 5 (unknown id 99),
# 6 (entry with no criterion_id -> discarded, not misattributed)
rawA = LLMEvaluationOutput(supplier_name='Apex', criteria=[
    CriterionResult(criterion_id=1, score=8,  justification='j', evidence='e'),
    CriterionResult(criterion_id=2, score=6,  justification='j', evidence='e'),
    CriterionResult(criterion_id=3, score=12, justification='j', evidence='e'),
    CriterionResult(criterion_id=99, score=5, justification='j', evidence='e'),
    CriterionResult(criterion_id=None, score=5, justification='j', evidence='e'),
    CriterionResult(criterion_id=5, score=0, justification='', evidence=''),
])
normA, warnA = normalize_llm_output(rawA, snap)
print('Apex normalized:', [(r.criterion_id, r.score) for r in normA])
for w in warnA:
    print('  -', w)

# Supplier B — clean
rawB = LLMEvaluationOutput(supplier_name='BrightPath', criteria=[
    CriterionResult(criterion_id=i, score=s, justification='j', evidence='e')
    for i, s in [(1, 10), (2, 4), (3, 8), (4, 10), (5, 0)]
])
normB, warnB = normalize_llm_output(rawB, snap)

# Edge case 4 — fully malformed response
normM, warnM = normalize_llm_output(None, snap)
assert [r.score for r in normM] == [0, 0, 0, 0, 0] and warnM == [MALFORMED_POLICY_WARNING]
print('\nMalformed policy:', warnM[0])

# Edge case 7 — criterion 5: every supplier scored 0 -> zero benchmark
per = {'Apex': normA, 'BrightPath': normB}
benchmarks, bench_warn = compute_benchmarks(per, snap)
print('\nbenchmarks:', benchmarks)
for w in bench_warn:
    print('  -', w)

scoredA, absA, ppiA = score_supplier(normA, snap, benchmarks)
scoredB, absB, ppiB = score_supplier(normB, snap, benchmarks)
print(f'\nApex:       absolute={absA:.2f}  ppi={ppiA:.2f}')
print(f'BrightPath: absolute={absB:.2f}  ppi={ppiB:.2f}')
for s in scoredA:
    print(f'  {s.criterion.name:22s} score={s.score:4.1f} bench={s.benchmark:4.1f} '
          f'gap={s.gap:5.1f} rel={s.relative_pct:6.1f}%')

# Hand-computed: absA = 8/10*30 + 6/10*20 + 10/10*20 = 56 ; ppiA = 80*.3+100*.2+100*.2+0*.2+100*.1 = 74
assert abs(absA - 56.0) < 1e-9 and abs(ppiA - 74.0) < 1e-9
assert abs(absB - 74.0) < 1e-9 and abs(ppiB - 89.333333333) < 1e-6
print('\nPhase 3 fixture matches hand computation.')

## Phase 4 — Ranking & Orchestration

`ranking.rank_suppliers` applies the fixed tie-break as a single stable composite
sort key `(-ppi, submission_date, -experience_rating, name.lower())`; failed
suppliers (`ppi is None`) are excluded entirely and never get a `final_rank`.

`orchestrator.execute_full_pipeline` chains Setup → Batch → Evaluate → Validate →
Score → Benchmark → Rank → Persist. The criterion snapshot is frozen once at
Batch time; `rfp_runs.status` always resolves to `completed`/`failed` in a
`finally` block.

The cell below exercises Edge Case Checklist items **8–11** with fixtures and a
stubbed pipeline (no API key / PDFs needed).

In [ ]:
from datetime import date

import document_tool, evaluation_agent, orchestrator
from models import SupplierScore, LLMEvaluationOutput
from ranking import rank_suppliers

snap4 = db.get_active_criteria()

def _ss(name, ppi, d, rating, failed=False):
    return SupplierScore(supplier_name=name, submission_date=d, experience_rating=rating,
                         criteria_snapshot=snap4, ppi=None if failed else ppi, failed=failed)

# 8 tie on PPI -> earlier date | 9 +date -> higher rating | 10 +rating -> name asc | 11 failed excluded
r8 = rank_suppliers([_ss('Beta', 90, date(2026,2,1), 3), _ss('Alpha', 90, date(2026,1,1), 3)])
r9 = rank_suppliers([_ss('Lo', 90, date(2026,1,1), 2), _ss('Hi', 90, date(2026,1,1), 5)])
r10 = rank_suppliers([_ss('zeta', 90, date(2026,1,1), 3), _ss('acme', 90, date(2026,1,1), 3)])
r11 = rank_suppliers([_ss('G1', 80, date(2026,1,1), 3), _ss('Bad', 0, date(2026,1,1), 3, failed=True),
                      _ss('G2', 95, date(2026,1,1), 3)])
print('8 :', [x.supplier_score.supplier_name for x in r8])
print('9 :', [x.supplier_score.supplier_name for x in r9])
print('10:', [x.supplier_score.supplier_name for x in r10])
print('11:', {x.supplier_score.supplier_name: x.final_rank for x in r11})
assert [x.supplier_score.supplier_name for x in r8] == ['Alpha', 'Beta']
assert [x.supplier_score.supplier_name for x in r9] == ['Hi', 'Lo']
assert [x.supplier_score.supplier_name for x in r10] == ['acme', 'zeta']
assert {x.supplier_score.supplier_name: x.final_rank for x in r11} == {'G2': 1, 'G1': 2, 'Bad': None}

# Stubbed full pipeline: swap the PDF + LLM steps, keep validate/score/benchmark/rank/persist real
document_tool.extract_text = lambda path, name: (_ for _ in ()).throw(
    document_tool.DocumentExtractionError(name, 'empty')) if 'broken' in path else f'text {name}'
def _fake_llm(name, text, criteria):
    s = {'Apex': 9, 'BrightPath': 5, 'NexaWorks': 7}.get(name, 6)
    return LLMEvaluationOutput(supplier_name=name, risks=['r'], overall_summary='ok',
        criteria=[CriterionResult(criterion_id=c.criterion_id, score=s, justification='j', evidence='e') for c in criteria])
evaluation_agent.evaluate_supplier = _fake_llm

meta = [{'name': n, 'submission_date': d, 'experience_rating': er} for n, d, er in [
    ('Apex', date(2026,1,3), 4.0), ('BrightPath', date(2026,1,1), 2.0),
    ('NexaWorks', date(2026,1,2), 5.0), ('Broken', date(2026,1,4), 3.0)]]
run_id, ranked = orchestrator.execute_full_pipeline(meta, ['a.pdf', 'b.pdf', 'c.pdf', 'broken.pdf'])
print('\nrun', run_id, '-> status', db.get_run(run_id)['status'])
for row in db.get_supplier_results(run_id):
    print(f"  {row['supplier_name']:12s} rank={row['final_rank']} ppi={row['ppi']} abs={row['absolute_score']}")
assert db.get_run(run_id)['status'] == 'completed'
assert sorted(x['final_rank'] for x in db.get_supplier_results(run_id) if x['final_rank']) == [1, 2, 3]
print('\nPhase 4 edge cases 8-11 + stubbed pipeline verified.')


## Phase 6 — Full pipeline demo + Edge Case Checklist

This section runs the **real** orchestrator end-to-end against the supplier PDFs in
`../data/sample_pdfs/` (needs `GEMINI_API_KEY` in `../.env`), then exercises the
deliberately-broken-input case. The 11 edge cases are demonstrated across the cells
above and summarised at the bottom.


In [ ]:
# Full successful run — real modules, real LLM, real persistence.
import glob
from datetime import date
from dotenv import load_dotenv

import orchestrator
load_dotenv(os.path.abspath('../.env'))

pdfs = sorted(glob.glob(os.path.abspath('../data/sample_pdfs/*.pdf')))
print('supplier PDFs:', [os.path.basename(p) for p in pdfs])
assert len(pdfs) >= 4, 'Add >=4 supplier PDFs to data/sample_pdfs/ first.'

# Submission dates / experience ratings are demo metadata (0-5 scale).
demo_meta = {
    'Apex Systems':    (date(2026, 1, 5), 4.5),
    'BrightPath Tech': (date(2026, 1, 2), 2.0),
    'NexaWorks':       (date(2026, 1, 4), 4.0),
    'Orbit Digital':   (date(2026, 1, 3), 4.5),
}
meta, paths = [], []
for p in pdfs:
    stem = os.path.splitext(os.path.basename(p))[0]
    name = next((k for k in demo_meta if k.lower().replace(' ', '') in stem.lower().replace(' ', '').replace('_', '')), stem)
    d, er = demo_meta.get(name, (date(2026, 1, 1), 3.0))
    meta.append({'name': name, 'submission_date': d, 'experience_rating': er})
    paths.append(p)

run_id, ranked = orchestrator.execute_full_pipeline(meta, paths)
print('\nrun', run_id, '-> status', db.get_run(run_id)['status'])
for row in db.get_supplier_results(run_id):
    print(f"  rank {str(row['final_rank']):>4}  {row['supplier_name']:16s} "
          f"abs={row['absolute_score']}  ppi={row['ppi']}")


In [ ]:
# Deliberately broken input (edge case 11): one corrupt PDF, rest of the batch still completes.
import tempfile

bad = os.path.join(tempfile.mkdtemp(), 'corrupt.pdf')
open(bad, 'wb').write(b'%PDF-1.4 this is not a real pdf body')

meta_b = meta[:2] + [{'name': 'Corrupt Co', 'submission_date': date(2026, 1, 9), 'experience_rating': 3.0}]
paths_b = paths[:2] + [bad]

run_id_b, _ = orchestrator.execute_full_pipeline(meta_b, paths_b)
print('run', run_id_b, '-> status', db.get_run(run_id_b)['status'])
for row in db.get_supplier_results(run_id_b):
    rj = row['result_json']
    print(f"  {row['supplier_name']:16s} rank={row['final_rank']} failed={rj.get('failed')}  warnings={rj.get('warnings')}")
assert db.get_run(run_id_b)['status'] == 'completed'
assert any(r['result_json'].get('failed') for r in db.get_supplier_results(run_id_b))
assert sorted(r['final_rank'] for r in db.get_supplier_results(run_id_b) if r['final_rank']) == [1, 2]
print('\nBroken-input case: corrupt supplier excluded (rank=None), rest of batch ranked 1..2.')


In [ ]:
# Export the successful run to sample_data/sample_run_result.json
# (same shape the Run Details "Download full run as JSON" button produces).
import json as _json

run = db.get_run(run_id)
rows = db.get_supplier_results(run_id)
ranked_rows  = [r for r in rows if r['final_rank'] is not None]
failed_rows  = [r for r in rows if r['final_rank'] is None]
snap = (ranked_rows + failed_rows)[0]['result_json']['criteria_snapshot']

export = {
    'rfp_run_id': run_id,
    'status': run['status'],
    'created_at': run['created_at'],
    'criterion_snapshot': snap,
    'ranked_suppliers':   [r['result_json'] for r in ranked_rows],
    'excluded_suppliers': [r['result_json'] for r in failed_rows],
}
out = os.path.abspath('../sample_data/sample_run_result.json')
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'w') as f:
    _json.dump(export, f, indent=2, default=str)
print('wrote', out)


## Edge Case Checklist — where each is demonstrated

| # | Case | Cell |
|---|---|---|
| 1 | Valid evaluation (happy path) | Phase 6 full-pipeline run |
| 2 | LLM response missing an active criterion -> defaulted to 0, warned | Phase 3 (`rawA`, criterion 4) |
| 3 | Out-of-range score -> clipped, warned | Phase 3 (`rawA`, criterion 3: 12 -> 10) |
| 4 | Fully malformed/unparseable JSON -> all criteria 0, warned | Phase 3 (`normalize_llm_output(None, snap)`) |
| 5 | Unknown `criterion_id` -> discarded, warned | Phase 3 (`rawA`, id 99) |
| 6 | Entry with no `criterion_id` -> discarded, not misattributed | Phase 3 (`rawA`, `criterion_id=None`) |
| 7 | Zero benchmark (all suppliers 0) -> 100% relative, warned | Phase 3 (criterion 5) |
| 8 | Tie on PPI -> earlier submission date | Phase 4 (`r8`) |
| 9 | Tie on PPI + date -> higher experience rating | Phase 4 (`r9`) |
| 10 | Full tie on PPI + date + rating -> name ascending | Phase 4 (`r10`) |
| 11 | Supplier PDF extraction fails -> excluded, rest of batch unaffected | Phase 4 (`r11` / stub) + Phase 6 broken-input cell |
